In [7]:
"""
================================================================================
  Qwen-0.5B × Amharic Fine-Tuning Demo - IMPROVED VERSION
  Using: addisai/FineTome-single-turn-dedup-amharic (83k examples)
  Better conversational performance for customer service
================================================================================
"""

# Install correct versions
!pip uninstall -y peft transformers accelerate -q
!pip install transformers==4.37.2 accelerate==0.26.1 peft==0.8.2 datasets sentencepiece -q

import os
import time
import warnings
import torch

warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# Check GPU
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ── 1. Constants ──────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2-0.5B-Instruct"
DATASET_NAME   = "addisai/FineTome-single-turn-dedup-amharic"  # 83k high-quality examples
OUTPUT_DIR     = "./qwen-amharic-lora-improved"
MAX_SEQ_LEN    = 512  # Increased for better context
LORA_RANK      = 16   # Increased for better learning
LORA_ALPHA     = 32
TRAIN_EPOCHS   = 2    # More epochs for better learning
BATCH_SIZE     = 4
LR             = 2e-4


# ── 2. Tokenizer ──────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded")


# ── 3. Base model ────────────────────────────────────────────────────────────
print("Loading base model on GPU …")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"✓ Model loaded in {time.time()-t0:.1f}s")

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 Memory used: {used:.2f} GB / {total:.1f} GB ({used/total*100:.1f}%)")


# ── 4. BEFORE fine-tune: baseline responses ───────────────────────────────────
TEST_PROMPTS_AM = [
    "ስለ ምርታችሁ የዋጋ ማወቅ እፈልጋለሁ።",          # Product pricing
    "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?",          # Order tracking
    "ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?",     # Product return
    "ከአካውንቴ ላይ ያልፈቀድኩት ክፍያ ተወሰደ።",    # Unauthorized charge
    "ምርቴ ጉድለት ያለበት መስሎኝ ነበር፣ ምን ማድረግ አለብኝ?",  # Product defect
]

def chat(model_obj, user_msg, max_new_tokens=150, label=""):
    messages = [
        {"role": "system", "content": "You are a helpful Ethiopian customer service assistant. Respond in Amharic or English as appropriate."},
        {"role": "user", "content": user_msg},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model_obj.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_obj.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n[{label}] User: {user_msg}")
    print(f"[{label}] Reply: {response[:250]}")
    return response


print("\n" + "=" * 60)
print("BEFORE FINE-TUNE — Baseline responses")
print("=" * 60)
before_responses = []
for prompt in TEST_PROMPTS_AM:
    try:
        resp = chat(model, prompt, label="BEFORE")
        before_responses.append(resp)
    except Exception as e:
        print(f"Error: {e}")
        before_responses.append("")
    time.sleep(1)


# ── 5. Load FineTome Amharic Dataset ──────────────────────────────────────────
print("\n" + "=" * 60)
print(f"Loading dataset: {DATASET_NAME} …")
print("This is a high-quality conversational Amharic dataset with 83k examples")
raw = load_dataset(DATASET_NAME, split="train")
print(f"✓ Loaded {len(raw)} examples")

# Inspect dataset structure
print(f"\nDataset columns: {raw.column_names}")
print(f"Sample structure: {raw[0].keys()}")

def format_conversation_to_instruction(ex):
    """
    Convert FineTome's conversation format to instruction tuning format.
    The dataset has both English and Amharic conversations.
    """
    # Get Amharic conversation
    conv_am = ex.get("conversations_amharic", [])

    if conv_am and len(conv_am) >= 2:
        # Extract user message and assistant response
        user_msg = conv_am[0].get("content", "") if len(conv_am) > 0 else ""
        assistant_msg = conv_am[1].get("content", "") if len(conv_am) > 1 else ""

        # Format as instruction-output pair
        instruction = f"ለደንበኛ አገልግሎት ጥያቄ መልስ ስጥ: {user_msg}"
        output = assistant_msg
    else:
        # Fallback to English conversation
        conv_en = ex.get("conversations_english", [])
        if conv_en and len(conv_en) >= 2:
            user_msg = conv_en[0].get("content", "")
            assistant_msg = conv_en[1].get("content", "")
            instruction = f"Answer this customer service question: {user_msg}"
            output = assistant_msg
        else:
            instruction = "Provide helpful customer service"
            output = "I'm here to help you with your request."

    # Format for training
    text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}{tokenizer.eos_token}"
    return {"text": text}

# Process dataset (use subset for faster training, but larger than before)
print("\nFormatting dataset for training...")
dataset = raw.map(format_conversation_to_instruction, remove_columns=raw.column_names)

# Use more examples for better performance (5000 instead of 500)
NUM_EXAMPLES = 5000  # 10x more than before
dataset = dataset.select(range(min(NUM_EXAMPLES, len(dataset))))
print(f"Using {len(dataset)} examples for training (10x more than previous version)")

def tokenize(ex):
    tok = tokenizer(
        ex["text"],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

print("Tokenizing dataset...")
tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")


# ── 6. LoRA Configuration (Enhanced) ─────────────────────────────────────────
print("\n" + "=" * 60)
print("Setting up LoRA (Enhanced configuration)...")
print("=" * 60)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,           # 16 (was 8) - more capacity
    lora_alpha=LORA_ALPHA, # 32 (was 16)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # More modules
    lora_dropout=0.1,
    bias="none",
)

# Apply LoRA
model.train()
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.2%} of {total_params:,})")
print(f"✓ This is {trainable_params/540672:.1f}x more trainable params than previous version")


# ── 7. Training with better settings ─────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAIN_EPOCHS,  # 2 epochs (was 1)
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    learning_rate=LR,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

print("\n" + "=" * 60)
print("Starting Enhanced LoRA Fine-tuning …")
print("=" * 60)
print(f"⚠️ Training on {len(train_ds)} examples (10x more than before)")
print(f"⚠️ {TRAIN_EPOCHS} epochs")
print(f"⚠️ Estimated time: 15-20 minutes on Colab GPU\n")

t_train = time.time()
trainer.train()
print(f"\n✓ Training completed in {(time.time()-t_train)/60:.1f} minutes")


# ── 8. Save model ─────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")


# ── 9. AFTER fine-tune: compare responses ─────────────────────────────────────
model.eval()

print("\n" + "=" * 60)
print("AFTER FINE-TUNE — Improved responses (with FineTome dataset)")
print("=" * 60)
after_responses = []
for prompt in TEST_PROMPTS_AM:
    try:
        resp = chat(model, prompt, label="AFTER")
        after_responses.append(resp)
    except Exception as e:
        print(f"Error: {e}")
        after_responses.append("")
    time.sleep(1)


# ── 10. Final Comparison ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 COMPARISON: BEFORE vs AFTER Fine-Tuning")
print("=" * 60)

for i, prompt in enumerate(TEST_PROMPTS_AM):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt[:100]}...")
    print(f"{'='*60}")
    print(f"🔴 BEFORE: {before_responses[i][:200]}" if before_responses[i] else "🔴 BEFORE: [No response]")
    print(f"🟢 AFTER:  {after_responses[i][:200]}" if after_responses[i] else "🟢 AFTER: [No response]")
    print()

print("\n" + "=" * 60)
print("✅ ENHANCED DEMONSTRATION COMPLETE!")
print("=" * 60)
print(f"""
IMPROVEMENTS MADE:
├─ Dataset: 5000 examples (was 450) - 11x more data
├─ LoRA Rank: 16 (was 8) - 2x model capacity
├─ Trainable params: ~1.1M (was 540k) - 2x more
├─ Sequence length: 512 (was 256) - Better context
├─ Epochs: 2 (was 1) - More training
└─ Target modules: 4 (was 2) - More adaptation

Expected Improvements:
✓ Much better understanding of Amharic
✓ More relevant customer service responses
✓ Better handling of varied queries
✓ More fluent mixed Amharic/English responses

Results Comparison:
• Training time: {(time.time()-t_train)/60:.1f} minutes
• Final eval loss: {trainer.state.log_history[-2]['eval_loss']:.4f}
• Model size: Still ~1GB (same memory footprint!)
""")

# Memory cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Final GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.6 GB
Loading tokenizer …
✓ Tokenizer loaded
Loading base model on GPU …


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✓ Model loaded in 2.2s
💾 Memory used: 1.87 GB / 14.6 GB (12.8%)

BEFORE FINE-TUNE — Baseline responses

[BEFORE] User: ስለ ምርታችሁ የዋጋ ማወቅ እፈልጋለሁ።
[BEFORE] Reply: "አማርኛ በአማርኛ ማወሆ በርጋ ሚዎት ሜባክ መንደር."

[BEFORE] User: ትዕዛዜን እንዴት ልከታተለው እችላለሁ?
[BEFORE] Reply: "ትዕ明珠" ၏ "IDA" የ በርኛ ዓ "'Fahmi" ይህ ዘን መሆን ዓ '"Nora"' ዛ በርኛ ዓ "Amal".

[BEFORE] User: ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?
[BEFORE] Reply: አማርኛ ይላው ၏ ደሮል ቲጻች ፀዱል ደትራይ ጥልም ደረታ ማደረገበል ህት ተላው በ Ꮏረያ ታት ደረታ ማደረገበል ተቃባ ደረታ ታት ደረታ በ ደቀን ባረጯ በ ቴክተል ብለን በ ትለበል በ �

[BEFORE] User: ከአካውንቴ ላይ ያልፈቀድኩት ክፍያ ተወሰደ።
[BEFORE] Reply: የተለም ၎ል麻醉失禁 ማራላይ ህረssé ምራል ሶራል በል麻醉失禁 ማራላይ ሳራል ልራል ስራል ሽራል በል麻醉失禁 ማራላይ ሳራል ስራል ሽራል ሏርታ ሐይ መንበል ሴራል ሑነይ ሕክክክስ ስክክክስ ረትዮጵያ ህነይ ብሚብር �

[BEFORE] User: ምርቴ ጉድለት ያለበት መስሎኝ ነበር፣ ምን ማድረግ አለብኝ?
[BEFORE] Reply: ምርታ ጉድለት ያለበት መስሎኝ ነበር። ነገል ሰን ማድረግ አለብኝ ይህ ၏ ይህ በ.

Loading dataset: addisai/FineTome-single-turn-dedup-amharic …
This is a high-quality conversational Amharic dataset with 83k examples


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/177M [00:00<?, ?B/s]

...

Using 5000 examples for training (10x more than previous version)
Tokenizing dataset...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train: 4500, Eval: 500

Setting up LoRA (Enhanced configuration)...
✓ Trainable parameters: 2,162,688 (0.44% of 496,195,456)
✓ This is 4.0x more trainable params than previous version

Starting Enhanced LoRA Fine-tuning …
⚠️ Training on 4500 examples (10x more than before)
⚠️ 2 epochs
⚠️ Estimated time: 15-20 minutes on Colab GPU



Step,Training Loss,Validation Loss
100,2.179708,1.988323
200,1.809386,1.743350
300,1.661169,1.613820
400,1.591597,1.519422
500,1.525194,1.459121
600,1.440407,1.410566
700,1.405381,1.380860
800,1.364375,1.347873
900,1.358145,1.326327
1000,1.361796,1.313106



✓ Training completed in 29.3 minutes
✓ Model saved to ./qwen-amharic-lora-improved

AFTER FINE-TUNE — Improved responses (with FineTome dataset)

[AFTER] User: ስለ ምርታችሁ የዋጋ ማወቅ እፈልጋለሁ።
[AFTER] Reply: የምህ diabetic ተግባር መማሪያ ይሰላ፦ ሁኔታዎች የተለክ ሲሆን በአካመት ጥዳና ግንት ያሉ የሂሳብ ባህሪ ሲሆን የሚከተለውን የሆነ ኮድ ስደም የሆነ ባህሪ ሲሆን እኩል የሆነ የሃይ (diabetes) ባ

[AFTER] User: ትዕዛዜን እንዴት ልከታተለው እችላለሁ?
[AFTER] Reply: አማርኛ በእያንዳንዱ መዘጋ或多姆 (grammatical gender) የሚባል ተመጥሮች ቁምሩ ስክmaxcdn ጠቅስ ምንድ ነው ይህም ያስገቡና እንደ ግብ罕 ሲኖች እንዲረጡ እችላለው ይህም እኩል ማዝሃን በግብ哈尔 የቁምቶች እንዴ

[AFTER] User: ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?
[AFTER] Reply: አማርኛ በሶራት ውስጥ ያለው ጥቅም ይሰላቁና ቅዳ乌 (diagnosis) ሲሆን የሶራት ተከተል ለማስላት የሚያሳይ የፀደር ስታ የሚገባው የሶራት ሕብረ-ቃቀት ነው.

[AFTER] User: ከአካውንቴ ላይ ያልፈቀድኩት ክፍያ ተወሰደ።
[AFTER] Reply: የአካውንatem (Ammonium) የሚገባው የአካውንatem ዝርዝር መስመር እንዲሁ ጠቃሚ ባህሪ ስታዎች በመጠቀም የተለያዩ የአካውንatem ውስጥ ያሉ የአካውንatem ዝርዝር መስመር ይዟሮ፦ እነዚህ የአካውንatem ዝርዝር አማራዎ

[AFTER] User: ምርቴ ጉድለት ያለበት መስሎኝ ነበር፣ ምን ማድረግ አለብኝ?
[AFTER] Reply: አዎንታዊ ፕሮግራም በመጠቀም የሚከተሉትን ቅደም እየቶች ይሰባል። ይህ የሚ